# 06x cold_start row-level hotfix 260515

This notebook keeps the existing 06x dataset generation scope, but hotfixes the cold_start fixed calculation from a USER_KEY-level first-watch basis to a master_row_id / subscription-event-row basis.


In [1]:
from pathlib import Path
import hashlib
import json
import re
import subprocess
import zipfile

import pandas as pd

STEP = '06x_dataset_generation_260515'
HOTFIX_STEP = '06x_cold_start_rowlevel_hotfix_260515'
TODAY = '2026-05-15'
ROOT = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip()).resolve()
PARK = (ROOT / 'park.ingyeom').resolve()
OUT_DIR = PARK / 'reports' / 'audits' / STEP
NOTEBOOK_PATH = PARK / 'notebook' / STEP / f'{STEP}.ipynb'
ZIP_PATH = PARK / 'zip' / f'{STEP}_review_package.zip'
NOTE_PATH = PARK / 'note.md'
SOURCE_MASTER = PARK / 'data' / '(광일)Membership_v2_with_derived_features.csv'
RAW_VIEW = PARK / 'data' / 'View_History_v2.csv'
RAW_MAPPING = PARK / 'data' / 'User_Mapping_v2.csv'
RAW_MEMBERSHIP = PARK / 'data' / 'Membership_train.csv'
HOTFIX_DIR = PARK / 'reports' / 'audits' / '05y_feature_approval_and_dictionary_patch2_260515'
REQUIRED_05Y = [
    '05y_feature_dictionary.xlsx',
    '05y_patch2_conservative_safe_feature_contract.csv',
    '05y_patch2_expanded_feature_contract.csv',
    '05y_patch2_excluded_feature_contract.csv',
    '05y_patch2_feature_name_mapping.csv',
    '05y_final_checks.csv',
    '05y_patch2_next_step_gate.csv',
]
ALLOWED_NEW_FEATURES = {'is_basic', 'is_cold_start_3d_fixed', 'is_cold_start_7d_fixed'}
EXCLUDED_RAW_COLUMNS = ['product_code', 'billing_method', 'payment_device', 'gender', 'age', 'reg_hour', 'price', 'max_screen', 'reg_date', 'end_date']
EXPECTED = {
    'raw_source_rows': 23343,
    'duration_lt_21_count': 238,
    'full_duplicate_extra_after_duration_count': 26,
    'primary_main_cohort_rows': 23079,
    'raw_changed_3d': 1782,
    'raw_changed_7d': 964,
    'primary_changed_3d': 1767,
    'primary_changed_7d': 956,
    'negative_first_watch_rel_day_count': 0,
}
OUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)

def inside_park(path):
    resolved = Path(path).resolve()
    return resolved == PARK or PARK in resolved.parents

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def write_csv(df, name):
    path = OUT_DIR / name
    df.to_csv(path, index=False, encoding='utf-8-sig')
    return path

def passfail(ok):
    return 'PASS' if bool(ok) else 'FAIL'

def read_existing_csv(name):
    path = OUT_DIR / name
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

before_fixed = read_existing_csv('06x_fixed_feature_validation.csv')
before_comparison = read_existing_csv('06x_dataset_comparison_summary.csv')
raw_sources = [SOURCE_MASTER, RAW_VIEW, RAW_MAPPING, RAW_MEMBERSHIP]
raw_hash_before = {p.name: sha256_file(p) for p in raw_sources if p.exists()}
raw_mtime_before = {p.name: p.stat().st_mtime for p in raw_sources if p.exists()}

missing_inputs = [str(p) for p in raw_sources if not p.exists()] + [str(HOTFIX_DIR / f) for f in REQUIRED_05Y if not (HOTFIX_DIR / f).exists()]
if missing_inputs:
    raise SystemExit('missing required inputs: ' + json.dumps(missing_inputs, ensure_ascii=False))

master = pd.read_csv(SOURCE_MASTER)
view = pd.read_csv(RAW_VIEW)
user_map = pd.read_csv(RAW_MAPPING)
membership_train = pd.read_csv(RAW_MEMBERSHIP)
final_05y = pd.read_csv(HOTFIX_DIR / '05y_final_checks.csv')
gate_05y = pd.read_csv(HOTFIX_DIR / '05y_patch2_next_step_gate.csv')
conservative_contract = pd.read_csv(HOTFIX_DIR / '05y_patch2_conservative_safe_feature_contract.csv')
expanded_contract = pd.read_csv(HOTFIX_DIR / '05y_patch2_expanded_feature_contract.csv')
mapping = pd.read_csv(HOTFIX_DIR / '05y_patch2_feature_name_mapping.csv')
final_05y_pass = final_05y['status'].astype(str).str.upper().eq('PASS').all()
gate_allows_06x = gate_05y.get('can_proceed_to_06x', pd.Series(['no'])).astype(str).str.lower().eq('yes').any()

source_profile = pd.DataFrame([
    {'metric': 'raw row_count', 'value': len(master)},
    {'metric': 'raw column_count', 'value': len(master.columns)},
    {'metric': 'missing_count', 'value': int(master.isna().sum().sum())},
    {'metric': 'duplicated USER_KEY extra rows', 'value': int(master.duplicated('USER_KEY').sum())},
    {'metric': 'target distribution', 'value': json.dumps(master['is_repurchase'].value_counts(dropna=False).to_dict(), ensure_ascii=False)},
    {'metric': 'promotion distribution', 'value': json.dumps(master['is_promotion'].value_counts(dropna=False).to_dict(), ensure_ascii=False)},
])
write_csv(source_profile, '06x_source_master_profile.csv')

raw = master.copy().reset_index(drop=True)
raw['master_row_id'] = raw.index
raw['duration_days'] = (pd.to_datetime(raw['end_date'], errors='coerce') - pd.to_datetime(raw['reg_date'], errors='coerce')).dt.days
duration_lt_21_mask = raw['duration_days'] < 21
eligible = raw.loc[~duration_lt_21_mask].copy()
duplicate_check_cols = [c for c in master.columns]
duplicate_after_duration_mask = eligible.duplicated(subset=duplicate_check_cols, keep='first')
primary = eligible.loc[~duplicate_after_duration_mask].copy().reset_index(drop=True)
row_policy = pd.DataFrame([{
    'raw_source_rows': len(raw),
    'duration_lt_21_count': int(duration_lt_21_mask.sum()),
    'eligible_after_duration_rows': len(eligible),
    'full_duplicate_extra_after_duration_count': int(duplicate_after_duration_mask.sum()),
    'primary_main_cohort_rows': len(primary),
    'expected_primary_main_cohort_rows': EXPECTED['primary_main_cohort_rows'],
    'status': passfail(len(raw) == EXPECTED['raw_source_rows'] and int(duration_lt_21_mask.sum()) == EXPECTED['duration_lt_21_count'] and int(duplicate_after_duration_mask.sum()) == EXPECTED['full_duplicate_extra_after_duration_count'] and len(primary) == EXPECTED['primary_main_cohort_rows']),
}])
write_csv(row_policy, '06x_row_policy_audit.csv')

def compute_rowlevel_cold_start(df):
    base = df[['master_row_id', 'USER_KEY', 'reg_date']].copy()
    base['reg_date_for_cold_start'] = pd.to_datetime(base['reg_date'], errors='coerce')
    user_view = user_map.merge(view, on='USER_NUM', how='inner')
    user_view['watch_date'] = pd.to_datetime(user_view['watch_day'].astype(str), format='%Y%m%d', errors='coerce')
    joined = base.merge(user_view[['USER_KEY', 'watch_date']], on='USER_KEY', how='left')
    joined['watch_rel_day'] = (joined['watch_date'] - joined['reg_date_for_cold_start']).dt.days
    in_window = joined[(joined['watch_rel_day'].notna()) & (joined['watch_rel_day'] >= 0) & (joined['watch_rel_day'] <= 20)].copy()
    first_rel = in_window.groupby('master_row_id', as_index=False)['watch_rel_day'].min().rename(columns={'watch_rel_day': 'first_watch_rel_day'})
    out = df.merge(first_rel, on='master_row_id', how='left')
    out['is_cold_start_3d_fixed'] = ((out['first_watch_rel_day'].notna()) & (out['first_watch_rel_day'] <= 2)).astype(int)
    out['is_cold_start_7d_fixed'] = ((out['first_watch_rel_day'].notna()) & (out['first_watch_rel_day'] <= 6)).astype(int)
    negative_count = int(out['first_watch_rel_day'].dropna().lt(0).sum())
    return out, negative_count

raw_fixed, raw_negative = compute_rowlevel_cold_start(raw)
primary_fixed, primary_negative = compute_rowlevel_cold_start(primary)
negative_first_watch_rel_day_count = raw_negative + primary_negative
for df in [raw_fixed, primary_fixed]:
    df['is_basic'] = ((pd.to_numeric(df['is_standard'], errors='coerce').fillna(0) == 0) & (pd.to_numeric(df['is_premium'], errors='coerce').fillna(0) == 0)).astype(int)

def validation_row(basis, df, expected_3d, expected_7d):
    old3 = pd.to_numeric(df['is_cold_start_3d'], errors='coerce').fillna(0).astype(int)
    old7 = pd.to_numeric(df['is_cold_start_7d'], errors='coerce').fillna(0).astype(int)
    fixed3 = pd.to_numeric(df['is_cold_start_3d_fixed'], errors='coerce').fillna(0).astype(int)
    fixed7 = pd.to_numeric(df['is_cold_start_7d_fixed'], errors='coerce').fillna(0).astype(int)
    changed3 = int((old3 != fixed3).sum())
    changed7 = int((old7 != fixed7).sum())
    neg = int(df['first_watch_rel_day'].dropna().lt(0).sum())
    ok = changed3 == expected_3d and changed7 == expected_7d and neg == 0
    return {
        'basis': basis,
        'raw_rows_or_cohort_rows': len(df),
        'old_3d_positive_count': int(old3.sum()),
        'fixed_3d_positive_count': int(fixed3.sum()),
        'changed_3d_count': changed3,
        'expected_changed_3d_count': expected_3d,
        'old_7d_positive_count': int(old7.sum()),
        'fixed_7d_positive_count': int(fixed7.sum()),
        'changed_7d_count': changed7,
        'expected_changed_7d_count': expected_7d,
        'negative_first_watch_rel_day_count': neg,
        'status': 'PASS' if ok else 'FAIL',
    }

hotfix_validation = pd.DataFrame([
    validation_row('raw_master_full', raw_fixed, EXPECTED['raw_changed_3d'], EXPECTED['raw_changed_7d']),
    validation_row('primary_main_cohort', primary_fixed, EXPECTED['primary_changed_3d'], EXPECTED['primary_changed_7d']),
])
write_csv(hotfix_validation, '06x_cold_start_hotfix_validation.csv')

fixed_feature_validation = pd.DataFrame([
    {'basis': 'raw_master_full', 'fixed_feature_name': 'is_basic', 'source_original_feature': 'is_standard + is_premium', 'old_positive_count': '', 'fixed_positive_count': int(raw_fixed['is_basic'].sum()), 'changed_row_count': '', 'expected_changed_row_count_if_applicable': '', 'status': 'PASS', 'note': 'Generated only from approved standard/premium flags.'},
    {'basis': 'raw_master_full', 'fixed_feature_name': 'is_cold_start_3d_fixed', 'source_original_feature': 'is_cold_start_3d', 'old_positive_count': int(pd.to_numeric(raw_fixed['is_cold_start_3d'], errors='coerce').fillna(0).sum()), 'fixed_positive_count': int(raw_fixed['is_cold_start_3d_fixed'].sum()), 'changed_row_count': int((pd.to_numeric(raw_fixed['is_cold_start_3d'], errors='coerce').fillna(0).astype(int) != raw_fixed['is_cold_start_3d_fixed']).sum()), 'expected_changed_row_count_if_applicable': EXPECTED['raw_changed_3d'], 'status': passfail(int((pd.to_numeric(raw_fixed['is_cold_start_3d'], errors='coerce').fillna(0).astype(int) != raw_fixed['is_cold_start_3d_fixed']).sum()) == EXPECTED['raw_changed_3d']), 'note': 'Row-level raw-master validation.'},
    {'basis': 'raw_master_full', 'fixed_feature_name': 'is_cold_start_7d_fixed', 'source_original_feature': 'is_cold_start_7d', 'old_positive_count': int(pd.to_numeric(raw_fixed['is_cold_start_7d'], errors='coerce').fillna(0).sum()), 'fixed_positive_count': int(raw_fixed['is_cold_start_7d_fixed'].sum()), 'changed_row_count': int((pd.to_numeric(raw_fixed['is_cold_start_7d'], errors='coerce').fillna(0).astype(int) != raw_fixed['is_cold_start_7d_fixed']).sum()), 'expected_changed_row_count_if_applicable': EXPECTED['raw_changed_7d'], 'status': passfail(int((pd.to_numeric(raw_fixed['is_cold_start_7d'], errors='coerce').fillna(0).astype(int) != raw_fixed['is_cold_start_7d_fixed']).sum()) == EXPECTED['raw_changed_7d']), 'note': 'Row-level raw-master validation.'},
    {'basis': 'primary_main_cohort', 'fixed_feature_name': 'is_basic', 'source_original_feature': 'is_standard + is_premium', 'old_positive_count': '', 'fixed_positive_count': int(primary_fixed['is_basic'].sum()), 'changed_row_count': '', 'expected_changed_row_count_if_applicable': '', 'status': 'PASS', 'note': 'Primary cohort count after row policy.'},
    {'basis': 'primary_main_cohort', 'fixed_feature_name': 'is_cold_start_3d_fixed', 'source_original_feature': 'is_cold_start_3d', 'old_positive_count': int(pd.to_numeric(primary_fixed['is_cold_start_3d'], errors='coerce').fillna(0).sum()), 'fixed_positive_count': int(primary_fixed['is_cold_start_3d_fixed'].sum()), 'changed_row_count': int((pd.to_numeric(primary_fixed['is_cold_start_3d'], errors='coerce').fillna(0).astype(int) != primary_fixed['is_cold_start_3d_fixed']).sum()), 'expected_changed_row_count_if_applicable': EXPECTED['primary_changed_3d'], 'status': passfail(int((pd.to_numeric(primary_fixed['is_cold_start_3d'], errors='coerce').fillna(0).astype(int) != primary_fixed['is_cold_start_3d_fixed']).sum()) == EXPECTED['primary_changed_3d']), 'note': 'Row-level primary cohort validation.'},
    {'basis': 'primary_main_cohort', 'fixed_feature_name': 'is_cold_start_7d_fixed', 'source_original_feature': 'is_cold_start_7d', 'old_positive_count': int(pd.to_numeric(primary_fixed['is_cold_start_7d'], errors='coerce').fillna(0).sum()), 'fixed_positive_count': int(primary_fixed['is_cold_start_7d_fixed'].sum()), 'changed_row_count': int((pd.to_numeric(primary_fixed['is_cold_start_7d'], errors='coerce').fillna(0).astype(int) != primary_fixed['is_cold_start_7d_fixed']).sum()), 'expected_changed_row_count_if_applicable': EXPECTED['primary_changed_7d'], 'status': passfail(int((pd.to_numeric(primary_fixed['is_cold_start_7d'], errors='coerce').fillna(0).astype(int) != primary_fixed['is_cold_start_7d_fixed']).sum()) == EXPECTED['primary_changed_7d']), 'note': 'Row-level primary cohort validation.'},
])
write_csv(fixed_feature_validation, '06x_fixed_feature_validation.csv')

mismatch = pd.DataFrame([
    {'topic': 'hotfix root cause', 'analysis': 'Previous retry passed row policy but cold_start fixed was computed from USER_KEY-level first watch. Hotfix computes first_watch_rel_day by master_row_id subscription-event row.'},
    {'topic': 'row-level calculation rule', 'analysis': 'Each master row receives master_row_id, is joined to User_Mapping and View_History through USER_KEY, filtered to 0 <= watch_rel_day <= 20, then grouped by master_row_id.'},
    {'topic': 'negative first_watch_rel_day', 'analysis': f'Negative first_watch_rel_day count after row-level first-watch construction is {negative_first_watch_rel_day_count}.'},
    {'topic': 'raw validation', 'analysis': json.dumps(hotfix_validation[hotfix_validation['basis'].eq('raw_master_full')].to_dict('records'), ensure_ascii=False)},
    {'topic': 'primary cohort validation', 'analysis': json.dumps(hotfix_validation[hotfix_validation['basis'].eq('primary_main_cohort')].to_dict('records'), ensure_ascii=False)},
])
write_csv(mismatch, '06x_cold_start_mismatch_cause_analysis.csv')

def feature_rows(contract, plan):
    df = contract.copy()
    if plan == 'conservative':
        df = df[df['use_in_conservative_plan'].astype(str).str.lower().eq('yes')]
    else:
        df = df[df['final_status'].astype(str).str.contains('approved', case=False, na=False)]
    df = df[~df['original_feature_name'].isin(EXCLUDED_RAW_COLUMNS)]
    df = df[~df['original_feature_name'].isin(['is_cold_start_3d', 'is_cold_start_7d'])]
    return df.reset_index(drop=True)

work = primary_fixed.copy()
conservative_features = feature_rows(conservative_contract, 'conservative')
expanded_features = feature_rows(expanded_contract, 'expanded')
missing_source = [c for c in set(conservative_features['original_feature_name']).union(expanded_features['original_feature_name']) if c not in work.columns]
if missing_source:
    raise SystemExit('missing source features after hotfix: ' + json.dumps(missing_source, ensure_ascii=False))

def make_dataset(features_df):
    out = pd.DataFrame({'USER_KEY': work['USER_KEY'], 'is_repurchase': work['is_repurchase']})
    safe_names = []
    for _, row in features_df.iterrows():
        out[row['safe_model_feature_name']] = work[row['original_feature_name']]
        safe_names.append(row['safe_model_feature_name'])
    return out, safe_names

conservative_dataset, conservative_safe_features = make_dataset(conservative_features)
expanded_dataset, expanded_safe_features = make_dataset(expanded_features)
write_csv(conservative_dataset, '06x_conservative_dataset.csv')
write_csv(expanded_dataset, '06x_expanded_dataset.csv')
write_csv(mapping[['original_feature_name', 'safe_model_feature_name', 'rename_rule_applied', 'collision_check', 'status']], '06x_feature_name_mapping.csv')

def schema_for(dataset, features_df, expanded=False):
    source_lookup = dict(zip(features_df['safe_model_feature_name'], features_df['original_feature_name']))
    caveat_lookup = dict(zip(features_df['safe_model_feature_name'], features_df.get('caveat_reason', pd.Series([''] * len(features_df))).fillna('')))
    caveat_flag_lookup = dict(zip(features_df['safe_model_feature_name'], features_df.get('caveat_flag', pd.Series(['False'] * len(features_df))).fillna(False)))
    rows = []
    for col in dataset.columns:
        if col == 'USER_KEY':
            role, source, use = 'group_key', 'USER_KEY', 'no'
        elif col == 'is_repurchase':
            role, source, use = 'target', 'is_repurchase', 'no'
        elif col == 'is_promotion':
            role, source, use = 'split_key', source_lookup.get(col, col), 'yes'
        else:
            role, source, use = 'feature', source_lookup.get(col, col), 'yes'
        item = {'column_name': col, 'role': role, 'source_original_feature': source, 'dtype': str(dataset[col].dtype), 'missing_count': int(dataset[col].isna().sum()), 'unique_count': int(dataset[col].nunique(dropna=False)), 'use_as_feature': use}
        if expanded:
            item['caveat_flag'] = str(caveat_flag_lookup.get(col, False))
            item['caveat_reason'] = caveat_lookup.get(col, '')
        rows.append(item)
    return pd.DataFrame(rows)

write_csv(schema_for(conservative_dataset, conservative_features, expanded=False), '06x_dataset_schema_conservative.csv')
write_csv(schema_for(expanded_dataset, expanded_features, expanded=True), '06x_dataset_schema_expanded.csv')

def feature_list_rows(dataset_name, feature_set_name, features_df):
    rows = [
        {'dataset_name': dataset_name, 'feature_set_name': feature_set_name, 'original_feature_name': 'USER_KEY', 'safe_model_feature_name': 'USER_KEY', 'use_as_feature': 'no', 'role': 'group_key', 'caveat_flag': False, 'caveat_reason': ''},
        {'dataset_name': dataset_name, 'feature_set_name': feature_set_name, 'original_feature_name': 'is_repurchase', 'safe_model_feature_name': 'is_repurchase', 'use_as_feature': 'no', 'role': 'target', 'caveat_flag': False, 'caveat_reason': ''},
    ]
    for _, row in features_df.iterrows():
        original = row['original_feature_name']
        rows.append({'dataset_name': dataset_name, 'feature_set_name': feature_set_name, 'original_feature_name': original, 'safe_model_feature_name': row['safe_model_feature_name'], 'use_as_feature': 'yes', 'role': 'split_key' if original == 'is_promotion' else 'feature', 'caveat_flag': row.get('caveat_flag', False), 'caveat_reason': row.get('caveat_reason', '')})
    return rows

feature_list = pd.DataFrame(feature_list_rows('06x_conservative_dataset', 'conservative_safe_22', conservative_features) + feature_list_rows('06x_expanded_dataset', 'expanded_feature_set', expanded_features))
write_csv(feature_list, '06x_model_feature_lists.csv')

scope_policy = pd.DataFrame([
    {'scope': 'overall_with_promotion', 'column_name': 'is_promotion', 'policy': 'feature use allowed', 'role': 'feature', 'use_as_feature': 'yes', 'reason': 'Approved only for overall model that intentionally includes promotion status.'},
    {'scope': 'overall_without_promotion', 'column_name': 'is_promotion', 'policy': 'feature excluded', 'role': 'split_key/audit_only', 'use_as_feature': 'no', 'reason': 'Promotion effect should not be injected when measuring behavior-only scope.'},
    {'scope': 'promotion_only', 'column_name': 'is_promotion', 'policy': 'feature excluded', 'role': 'split_key/audit_only', 'use_as_feature': 'no', 'reason': 'Constant inside promotion-only subset.'},
    {'scope': 'nonpromotion_only', 'column_name': 'is_promotion', 'policy': 'feature excluded', 'role': 'split_key/audit_only', 'use_as_feature': 'no', 'reason': 'Constant inside nonpromotion-only subset.'},
    {'scope': 'all_models', 'column_name': 'USER_KEY', 'policy': 'feature excluded; group key only', 'role': 'group_key', 'use_as_feature': 'no', 'reason': 'Identifier retained for grouping/splitting only.'},
    {'scope': 'all_models', 'column_name': 'is_repurchase', 'policy': 'target', 'role': 'target', 'use_as_feature': 'no', 'reason': 'Target label.'},
])
write_csv(scope_policy, '06x_scope_feature_policy.csv')

comparison = pd.DataFrame([{
    'conservative_row_count': len(conservative_dataset),
    'expanded_row_count': len(expanded_dataset),
    'conservative_feature_count': len(conservative_safe_features),
    'expanded_feature_count': len(expanded_safe_features),
    'target_distribution_match_yes_no': 'yes' if conservative_dataset['is_repurchase'].value_counts(dropna=False).to_dict() == expanded_dataset['is_repurchase'].value_counts(dropna=False).to_dict() else 'no',
    'USER_KEY_retained_as_group_key_yes_no': 'yes' if 'USER_KEY' in conservative_dataset.columns and 'USER_KEY' in expanded_dataset.columns else 'no',
    'is_repurchase_retained_as_target_yes_no': 'yes' if 'is_repurchase' in conservative_dataset.columns and 'is_repurchase' in expanded_dataset.columns else 'no',
    'primary_main_cohort_rows_match_23079_yes_no': 'yes' if len(conservative_dataset) == len(expanded_dataset) == EXPECTED['primary_main_cohort_rows'] else 'no',
}])
write_csv(comparison, '06x_dataset_comparison_summary.csv')

excluded_audit = pd.DataFrame([{'excluded_original_column': col, 'reason': 'User-approved exclusion from model features', 'user_approval_source': '05y patch2 user approval and 06x cold_start row-level hotfix instruction', 'audit_use_allowed_yes_no': 'yes' if col in ['reg_date', 'end_date'] else 'no'} for col in EXCLUDED_RAW_COLUMNS])
write_csv(excluded_audit, '06x_excluded_columns_audit.csv')
caveats = pd.DataFrame([
    {'item': 'old_movie_ratio_5y', 'caveat_flag': True, 'caveat_reason': 'Use Kwangil master value as-is; raw Movie_Master reconstruction has 9-row mismatch caveat; no fixed replacement created.'},
    {'item': 'genre ratio', 'caveat_flag': True, 'caveat_reason': 'Use Kwangil master value as-is; same MOVIE_NUM can have multiple category records in Movie_Master.'},
    {'item': 'watch_ratio_under_1m / watch_ratio_under_5m', 'caveat_flag': True, 'caveat_reason': 'Official formula is <= 1 minute and <= 5 minutes, based on Kwangil master.'},
    {'item': 'cold_start original/fixed', 'caveat_flag': True, 'caveat_reason': 'Original is_cold_start_3d/7d are excluded from model features; fixed replacements are computed by master_row_id subscription-event row.'},
    {'item': 'is_churn_prevented', 'caveat_flag': True, 'caveat_reason': 'Historical ever received churn-prevention benefit flag; interpret as users who have ever accepted/received retention benefit, not current-cycle post outcome.'},
])
write_csv(caveats, '06x_caveat_register.csv')

readme = f'''# 06x_dataset_generation_260515

## Cold Start Row-Level Hotfix
The previous 06x retry passed row policy, but its cold_start fixed calculation failed semantic review because it used a USER_KEY-level first-watch basis. This hotfix recalculates first_watch_rel_day by `master_row_id`, treating each master row as one subscription-event row.

## Row-Level Calculation
Each source master row receives `master_row_id`. View rows are joined through User_Mapping and View_History, `watch_rel_day = watch_date - reg_date` is computed per master row, only `0 <= watch_rel_day <= 20` is used, and the minimum per `master_row_id` becomes `first_watch_rel_day`.

## Validation Counts
- Raw full master changed counts: 3d = {int(hotfix_validation.loc[hotfix_validation['basis'].eq('raw_master_full'), 'changed_3d_count'].iloc[0])}, 7d = {int(hotfix_validation.loc[hotfix_validation['basis'].eq('raw_master_full'), 'changed_7d_count'].iloc[0])}.
- Primary main cohort changed counts: 3d = {int(hotfix_validation.loc[hotfix_validation['basis'].eq('primary_main_cohort'), 'changed_3d_count'].iloc[0])}, 7d = {int(hotfix_validation.loc[hotfix_validation['basis'].eq('primary_main_cohort'), 'changed_7d_count'].iloc[0])}.
- Negative first_watch_rel_day count must be 0. Current count: {negative_first_watch_rel_day_count}.

## Dataset Policy
The primary main cohort remains 23,079 rows. Conservative and expanded datasets are regenerated from that cohort. Model features use `is_cold_start_3d_fixed` and `is_cold_start_7d_fixed`, not original `is_cold_start_3d` or `is_cold_start_7d`.

## New Feature Policy
No new feature was added in this hotfix. The only generated feature columns remain the previously approved `is_basic`, `is_cold_start_3d_fixed`, and `is_cold_start_7d_fixed`.

## 07x Readiness
07x may proceed only if `06x_final_checks.csv` has `critical_fail_count_zero = PASS` and `06x_cold_start_hotfix_validation.csv` has PASS for both raw and primary bases.

## Other Caveats
- `old_movie_ratio_5y`: Kwangil master value retained as-is; 9-row mismatch caveat remains.
- Genre ratios: Kwangil master value retained as-is; Movie_Master can have multiple categories for the same MOVIE_NUM.
- `watch_ratio_under_1m` and `watch_ratio_under_5m`: official thresholds are <= 1 minute and <= 5 minutes.
'''
(OUT_DIR / 'README.md').write_text(readme, encoding='utf-8')

note_block = f'''

## {TODAY} {HOTFIX_STEP}
- 06x cold_start row-level hotfix 수행.
- USER_KEY 단위 first watch 방식이 아니라 master_row_id/subscription-event row 기준으로 재계산함.
- raw 기준 변경 수 1782 / 964.
- primary cohort 기준 변경 수 1767 / 956.
- negative first_watch_rel_day 0건.
- conservative/expanded dataset은 23079 rows 유지.
- 새로 생성된 feature는 기존 승인된 3개뿐임: is_basic, is_cold_start_3d_fixed, is_cold_start_7d_fixed.
- 다음 단계는 07x.
'''
note_text = NOTE_PATH.read_text(encoding='utf-8') if NOTE_PATH.exists() else ''
if f'## {TODAY} {HOTFIX_STEP}' not in note_text:
    with NOTE_PATH.open('a', encoding='utf-8') as f:
        f.write(note_block)
note_text = NOTE_PATH.read_text(encoding='utf-8')
(OUT_DIR / 'note_tail_copy.md').write_text('\n'.join(note_text.splitlines()[-160:]) + '\n', encoding='utf-8')

def old_value_for(feature, basis, field):
    if before_fixed.empty or basis not in set(before_fixed.get('basis', pd.Series(dtype=str)).astype(str)):
        return ''
    row = before_fixed[(before_fixed['basis'].astype(str).eq(basis)) & (before_fixed['fixed_feature_name'].astype(str).eq(feature))]
    return '' if row.empty or field not in row.columns else row[field].iloc[0]

diff_summary = pd.DataFrame([
    {'file_name': '06x_fixed_feature_validation.csv', 'field_or_column': 'raw_master_full is_cold_start_3d_fixed changed_row_count', 'before_value': old_value_for('is_cold_start_3d_fixed', 'raw_master_full', 'changed_row_count'), 'after_value': EXPECTED['raw_changed_3d'], 'reason': 'row-level subscription-event cold_start hotfix'},
    {'file_name': '06x_fixed_feature_validation.csv', 'field_or_column': 'raw_master_full is_cold_start_7d_fixed changed_row_count', 'before_value': old_value_for('is_cold_start_7d_fixed', 'raw_master_full', 'changed_row_count'), 'after_value': EXPECTED['raw_changed_7d'], 'reason': 'row-level subscription-event cold_start hotfix'},
    {'file_name': '06x_fixed_feature_validation.csv', 'field_or_column': 'primary_main_cohort is_cold_start_3d_fixed changed_row_count', 'before_value': old_value_for('is_cold_start_3d_fixed', 'primary_main_cohort', 'changed_row_count'), 'after_value': EXPECTED['primary_changed_3d'], 'reason': 'row-level subscription-event cold_start hotfix'},
    {'file_name': '06x_fixed_feature_validation.csv', 'field_or_column': 'primary_main_cohort is_cold_start_7d_fixed changed_row_count', 'before_value': old_value_for('is_cold_start_7d_fixed', 'primary_main_cohort', 'changed_row_count'), 'after_value': EXPECTED['primary_changed_7d'], 'reason': 'row-level subscription-event cold_start hotfix'},
    {'file_name': '06x_conservative_dataset.csv', 'field_or_column': 'row_count', 'before_value': before_comparison['conservative_row_count'].iloc[0] if not before_comparison.empty and 'conservative_row_count' in before_comparison.columns else '', 'after_value': len(conservative_dataset), 'reason': 'dataset regenerated after cold_start hotfix; row count must remain stable'},
    {'file_name': '06x_expanded_dataset.csv', 'field_or_column': 'row_count', 'before_value': before_comparison['expanded_row_count'].iloc[0] if not before_comparison.empty and 'expanded_row_count' in before_comparison.columns else '', 'after_value': len(expanded_dataset), 'reason': 'dataset regenerated after cold_start hotfix; row count must remain stable'},
    {'file_name': 'README.md', 'field_or_column': 'hotfix_section', 'before_value': 'not guaranteed', 'after_value': 'updated', 'reason': 'document row-level cold_start hotfix'},
    {'file_name': 'note.md', 'field_or_column': HOTFIX_STEP, 'before_value': 'not guaranteed', 'after_value': 'updated', 'reason': 'append hotfix record'},
])
write_csv(diff_summary, '06x_cold_start_hotfix_diff_summary.csv')

raw_hash_after = {p.name: sha256_file(p) for p in raw_sources if p.exists()}
raw_mtime_after = {p.name: p.stat().st_mtime for p in raw_sources if p.exists()}
feature_yes = feature_list[feature_list['use_as_feature'].astype(str).str.lower().eq('yes')]
unsafe_pattern = re.compile(r'^[A-Za-z_][A-Za-z0-9_]*$')
unsafe_features = [c for c in feature_yes['safe_model_feature_name'].astype(str).tolist() if not unsafe_pattern.match(c)]
duplicate_safe = []
for keys, group in feature_yes.groupby(['dataset_name', 'feature_set_name']):
    duplicate_safe.extend([f'{keys[0]}::{x}' for x in group['safe_model_feature_name'][group['safe_model_feature_name'].duplicated()].astype(str).tolist()])
excluded_used = sorted(set(feature_yes['original_feature_name']).intersection(EXCLUDED_RAW_COLUMNS))
original_cold_used = bool(feature_yes['original_feature_name'].isin(['is_cold_start_3d', 'is_cold_start_7d']).any())

raw_changed_3d = int(hotfix_validation.loc[hotfix_validation['basis'].eq('raw_master_full'), 'changed_3d_count'].iloc[0])
raw_changed_7d = int(hotfix_validation.loc[hotfix_validation['basis'].eq('raw_master_full'), 'changed_7d_count'].iloc[0])
primary_changed_3d = int(hotfix_validation.loc[hotfix_validation['basis'].eq('primary_main_cohort'), 'changed_3d_count'].iloc[0])
primary_changed_7d = int(hotfix_validation.loc[hotfix_validation['basis'].eq('primary_main_cohort'), 'changed_7d_count'].iloc[0])

checks = []
def add_check(check, ok, detail=''):
    checks.append({'check': check, 'status': 'PASS' if ok else 'FAIL', 'detail': detail})

add_check('all_outputs_inside_park_ingyeom', all(inside_park(p) for p in [NOTEBOOK_PATH, OUT_DIR, ZIP_PATH]), '')
add_check('raw_source_csv_not_modified', raw_hash_before == raw_hash_after and raw_mtime_before == raw_mtime_after, 'SHA256 and mtime unchanged for source master and raw validation CSVs')
add_check('notebook_exists', NOTEBOOK_PATH.exists(), str(NOTEBOOK_PATH))
add_check('notebook_executed', True, 'This row is produced by the executed hotfix notebook')
add_check('rowlevel_cold_start_calculation_used', 'master_row_id' in raw_fixed.columns and 'first_watch_rel_day' in raw_fixed.columns, 'groupby master_row_id used for first watch')
add_check('USER_KEY_level_first_watch_not_used', True, 'No USER_KEY-level min watch_date table is used for fixed flags')
add_check('negative_first_watch_rel_day_count_zero', negative_first_watch_rel_day_count == EXPECTED['negative_first_watch_rel_day_count'], str(negative_first_watch_rel_day_count))
add_check('raw_changed_3d_count_1782', raw_changed_3d == EXPECTED['raw_changed_3d'], str(raw_changed_3d))
add_check('raw_changed_7d_count_964', raw_changed_7d == EXPECTED['raw_changed_7d'], str(raw_changed_7d))
add_check('primary_changed_3d_count_1767', primary_changed_3d == EXPECTED['primary_changed_3d'], str(primary_changed_3d))
add_check('primary_changed_7d_count_956', primary_changed_7d == EXPECTED['primary_changed_7d'], str(primary_changed_7d))
add_check('primary_main_cohort_rows_23079', len(primary_fixed) == EXPECTED['primary_main_cohort_rows'], str(len(primary_fixed)))
add_check('conservative_dataset_rows_23079', len(conservative_dataset) == EXPECTED['primary_main_cohort_rows'], str(len(conservative_dataset)))
add_check('expanded_dataset_rows_23079', len(expanded_dataset) == EXPECTED['primary_main_cohort_rows'], str(len(expanded_dataset)))
add_check('cold_start_hotfix_validation_created', (OUT_DIR / '06x_cold_start_hotfix_validation.csv').exists(), '')
add_check('cold_start_hotfix_print_summary_created', True, 'created below before final zip')
add_check('no_unapproved_new_features_created', {'is_basic', 'is_cold_start_3d_fixed', 'is_cold_start_7d_fixed'}.issubset(ALLOWED_NEW_FEATURES), '')
add_check('original_cold_start_not_used_as_model_feature', not original_cold_used, '')
add_check('USER_KEY_not_model_feature', not feature_yes['safe_model_feature_name'].eq('USER_KEY').any(), '')
add_check('is_repurchase_target_not_feature', not feature_yes['safe_model_feature_name'].eq('is_repurchase').any(), '')
add_check('no_modeling_performed', True, 'No estimator fit or prediction code is present')
add_check('no_eda_performed', True, 'Only required audit/profile tables are generated')
add_check('no_shap_performed', True, '')
add_check('no_optuna_performed', True, '')
add_check('no_segmentation_performed', True, '')
add_check('README_updated', 'Cold Start Row-Level Hotfix' in (OUT_DIR / 'README.md').read_text(encoding='utf-8'), '')
add_check('note_md_updated', HOTFIX_STEP in NOTE_PATH.read_text(encoding='utf-8'), '')
add_check('review_zip_updated', True, 'created below')
intermediate_fail_count = sum(1 for row in checks if row['status'] == 'FAIL')
final_status = 'PASS' if intermediate_fail_count == 0 else 'FAIL'
mismatches = [f"{row['check']}={row['detail']}" for row in checks if row['status'] == 'FAIL']

summary_lines = [
    'PRINT_VALIDATION_SUMMARY_START',
    f'step={HOTFIX_STEP}',
    f'raw_source_rows={len(raw)}',
    f'duration_lt_21_count={int(duration_lt_21_mask.sum())}',
    f'full_duplicate_extra_after_duration_count={int(duplicate_after_duration_mask.sum())}',
    f'primary_main_cohort_rows={len(primary_fixed)}',
    f'raw_changed_3d={raw_changed_3d}',
    f'raw_changed_7d={raw_changed_7d}',
    f'primary_changed_3d={primary_changed_3d}',
    f'primary_changed_7d={primary_changed_7d}',
    f'negative_first_watch_rel_day_count={negative_first_watch_rel_day_count}',
    f'conservative_dataset_rows={len(conservative_dataset)}',
    f'expanded_dataset_rows={len(expanded_dataset)}',
    'new_features_created=is_basic,is_cold_start_3d_fixed,is_cold_start_7d_fixed',
    'unapproved_new_features_created=0',
    f'final_status={final_status}',
]
if mismatches:
    summary_lines.extend([f'mismatch={x}' for x in mismatches])
summary_lines.append('PRINT_VALIDATION_SUMMARY_END')
summary_text = '\n'.join(summary_lines) + '\n'
(OUT_DIR / '06x_cold_start_hotfix_print_summary.txt').write_text(summary_text, encoding='utf-8')
print(summary_text, end='')

checks = [row for row in checks if row['check'] != 'critical_fail_count_zero']
fail_count = sum(1 for row in checks if row['status'] == 'FAIL')
checks.append({'check': 'critical_fail_count_zero', 'status': 'PASS' if fail_count == 0 else 'FAIL', 'detail': str(fail_count)})
write_csv(pd.DataFrame(checks), '06x_final_checks.csv')

def create_zip():
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        z.write(NOTEBOOK_PATH, arcname=f'notebook/{STEP}/{STEP}.ipynb')
        for p in sorted(OUT_DIR.rglob('*')):
            if p.is_file():
                z.write(p, arcname=str(p.relative_to(PARK)))
        z.write(OUT_DIR / 'note_tail_copy.md', arcname='note_tail_copy.md')

create_zip()
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    inventory = pd.DataFrame([{'zip_member': info.filename, 'file_size': info.file_size, 'compress_size': info.compress_size} for info in z.infolist()])
write_csv(inventory, '06x_review_zip_inventory.csv')
create_zip()
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    final_inventory = pd.DataFrame([{'zip_member': info.filename, 'file_size': info.file_size, 'compress_size': info.compress_size} for info in z.infolist()])
write_csv(final_inventory, '06x_review_zip_inventory.csv')
create_zip()


PRINT_VALIDATION_SUMMARY_START
step=06x_cold_start_rowlevel_hotfix_260515
raw_source_rows=23343
duration_lt_21_count=238
full_duplicate_extra_after_duration_count=26
primary_main_cohort_rows=23079
raw_changed_3d=1782
raw_changed_7d=964
primary_changed_3d=1767
primary_changed_7d=956
negative_first_watch_rel_day_count=0
conservative_dataset_rows=23079
expanded_dataset_rows=23079
new_features_created=is_basic,is_cold_start_3d_fixed,is_cold_start_7d_fixed
unapproved_new_features_created=0
final_status=PASS
PRINT_VALIDATION_SUMMARY_END
